In [1]:
import sys
sys.path.append('..')

In [2]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import ParameterGrid

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [5]:
# Linf
# Grid Search

def getStats(x: np.ndarray, theta: np.ndarray, x0: np.ndarray, lamb):
    return np.log(1 + np.exp(-(x @ theta))) + \
        (lamb * (np.linalg.norm(x0 - x, ord=1)))

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.hstack((weights_adv, bias_adv))

def getAllPossibleX(x0: np.ndarray, length=6, step_size=0.1):
    # x0 don't have bias
    x0 = np.hstack((x0, np.array([1])))
    d = [[(length - 1) * -abs(x), length * abs(x)] for x in x0]
    d[x0.size- 1][0] = 1
    d[x0.size- 1][1] = 1

    delta_x = [np.arange(d[i][0], d[i][1] + step_size/2, step_size) for i in range(len(x0))]
    X = np.array(np.meshgrid(*delta_x)).T.reshape(-1, x0.shape[0])
    return np.round(X, decimals=5)

def searchGrid_linf(X, x0_withBias, theta0, bias0, alpha, lamb):
    Js = np.apply_along_axis(lambda x : getStats(x, calThetaAdv_linf(x[:-1], theta0, bias0, alpha), x0_withBias, lamb), arr=X, axis=1)
    Js_min_i = np.argmin(Js)
    xR_new_GS = X[Js_min_i]

    return xR_new_GS[:-1]

In [ ]:
# GS algorithm

alphas = np.linspace(0.1, 0.8, 8).round(3)      # <------------- Don't Touch
sba_l1psd_seed = 7              # <------------- Don't Touch
clf = "lr"
datasets = ["synthesis"]
lambdas = [0.1, 0.2, 0.3]       # <-------------
save_file = False
should_sample = False
divide_by_norm = False

func_name = "GS"

for dataset in datasets:
    # Read File and Get results
    # file_path = f"../results/cost_validity_latest/{dataset}_correctRELU.pickle"
    file_path = f"../results/cost_validity_latest/lr_{dataset}_alg1_lamb0.1.pickle"
    
    ret = pd.read_pickle(file_path)

    x0s = np.array(ret['x_0'])
    x0s = x0s[0]
    if should_sample:
        rng = np.random.default_rng(seed=sba_l1psd_seed)
        size_N = int(np.rint(0.15 * x0s.shape[0]))
        x0s = rng.choice(x0s, size=size_N, replace=False)

    try:
        theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
        bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 
    except TypeError:
        try: 
            theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64)
        except IndexError:
            theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][1].numpy().astype(np.float64)
    if divide_by_norm:
        divider = np.linalg.norm(theta0, 2)
        theta0 = theta0 / divider
        bias0 = bias0 / divider

    # ret['x_0'] = ret['x_0'][:len(alphas)]
    x0s_output = []
    for i in range(len(alphas)):
        x0s_output.append(deepcopy(x0s))
    ret['x_0'] = x0s_output

    # filtered_paramVal = [val['delta_max'] for val in ret['params'] if val['delta_max'] in alphas]
    filtered_paramVal = [val for val in alphas]
    ret['params'] = ParameterGrid({'delta_max': filtered_paramVal})

    for lamb in lambdas:
        xRs = [] # Final recouuse list

        if should_sample:
            for i in range(len(ret['x_0'])):
                ret['x_0'][i] = x0s 

        for alpha in alphas:
            res = []

            for x0 in tqdm.tqdm(x0s, desc=f"Running {func_name} on {clf}_{dataset} with lambda = {lamb}, alpha = {alpha}"):
                x0_withBias = np.hstack((x0, np.array([1])))
                X = getAllPossibleX(x0, length=5, step_size=0.5)
                res.append(searchGrid_linf(X, x0_withBias, theta0, bias0, alpha, lamb))
            xRs.append(res)
        
        ret['x_r'] = xRs
        
        # Save each (dataset, recourse function, lambda) file
        if save_file:
            file_path_saved = f"../results/cost_validity_latest/{clf}_{dataset}_{func_name}_lamb{lamb}_new.pickle"
            with open(file_path_saved, 'wb') as outFile:
                pickle.dump(ret, outFile)
                print(f"{clf}_{dataset}_{func_name}_lamb{lamb}_new.pickle is saved!")

Running lr_synthesis with lambda = 0.1, alpha = 0.1: 100%|██████████| 100/100 [00:02<00:00, 48.57it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.2:  84%|████████▍ | 84/100 [00:01<00:00, 48.41it/s]


KeyboardInterrupt: 

In [ ]:
alphas = np.linspace(0.1, 0.8, 8).round(3)      # <------------- Don't Touch
sba_l1psd_seed = 7              # <------------- Don't Touch
clf = "lr"
datasets = ["synthesis"]
recourse_fns = [ROAR]    # <---------------------
lambdas = [0.1, 0.2, 0.3]       # <-------------
save_file = False
should_sample = False
divide_by_norm = False


for dataset in datasets:
    # Read File and Get results
    # file_path = f"../results/cost_validity_latest/{dataset}_correctRELU.pickle"
    file_path = f"../results/cost_validity_latest/lr_{dataset}_alg1_lamb0.1.pickle"
    
    ret = pd.read_pickle(file_path)

    x0s = np.array(ret['x_0'])
    x0s = x0s[0]
    if should_sample:
        rng = np.random.default_rng(seed=sba_l1psd_seed)
        size_N = int(np.rint(0.15 * x0s.shape[0]))
        x0s = rng.choice(x0s, size=size_N, replace=False)

    try:
        theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
        bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 
    except TypeError:
        try: 
            theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64)
        except IndexError:
            theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][1].numpy().astype(np.float64)
    if divide_by_norm:
        divider = np.linalg.norm(theta0, 2)
        theta0 = theta0 / divider
        bias0 = bias0 / divider

    # ret['x_0'] = ret['x_0'][:len(alphas)]
    x0s_output = []
    for i in range(len(alphas)):
        x0s_output.append(deepcopy(x0s))
    ret['x_0'] = x0s_output

    # filtered_paramVal = [val['delta_max'] for val in ret['params'] if val['delta_max'] in alphas]
    filtered_paramVal = [val for val in alphas]
    ret['params'] = ParameterGrid({'delta_max': filtered_paramVal})

    for recourse_fn in recourse_fns:
        for lamb in lambdas:
            xRs = [] # Final recouuse list

            if should_sample:
                for i in range(len(ret['x_0'])):
                    ret['x_0'][i] = x0s 

            for alpha in alphas:
                res = []
                reco = recourse_fn(weights=theta0, bias=bias0, alpha=alpha, lamb=lamb)

                for x0 in tqdm.tqdm(x0s, desc=f"Running {reco.name} on {clf}_{dataset} with lambda = {lamb}, alpha = {alpha}"):
                    res.append(reco.get_recourse(x0))
                xRs.append(res)
            
            ret['x_r'] = xRs
            
            # Save each (dataset, recourse function, lambda) file
            if save_file:
                file_path_saved = f"../results/cost_validity_latest/{clf}_{dataset}_{reco.name}_lamb{lamb}_new.pickle"
                with open(file_path_saved, 'wb') as outFile:
                    pickle.dump(ret, outFile)
                    print(f"{clf}_{dataset}_{reco.name}_lamb{lamb}_new.pickle is saved!")

Running lr_synthesis with lambda = 0.1, alpha = 0.1: 100%|██████████| 100/100 [01:27<00:00,  1.14it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.2: 100%|██████████| 100/100 [01:15<00:00,  1.33it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.3: 100%|██████████| 100/100 [01:18<00:00,  1.28it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.4: 100%|██████████| 100/100 [01:07<00:00,  1.48it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.5: 100%|██████████| 100/100 [00:52<00:00,  1.89it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.6: 100%|██████████| 100/100 [00:49<00:00,  2.03it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.7: 100%|██████████| 100/100 [00:48<00:00,  2.06it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.8: 100%|██████████| 100/100 [00:47<00:00,  2.09it/s]


lr_synthesis_ROARL1_lamb0.1_new.pickle is saved!


Running lr_synthesis with lambda = 0.2, alpha = 0.1: 100%|██████████| 100/100 [00:50<00:00,  1.98it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.2: 100%|██████████| 100/100 [01:08<00:00,  1.46it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.3: 100%|██████████| 100/100 [01:03<00:00,  1.57it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.4: 100%|██████████| 100/100 [00:53<00:00,  1.85it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.5: 100%|██████████| 100/100 [00:53<00:00,  1.87it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.6: 100%|██████████| 100/100 [00:50<00:00,  2.00it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.7: 100%|██████████| 100/100 [00:48<00:00,  2.04it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.8: 100%|██████████| 100/100 [00:48<00:00,  2.07it/s]


lr_synthesis_ROARL1_lamb0.2_new.pickle is saved!


Running lr_synthesis with lambda = 0.3, alpha = 0.1: 100%|██████████| 100/100 [00:43<00:00,  2.31it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.2: 100%|██████████| 100/100 [00:46<00:00,  2.13it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.3: 100%|██████████| 100/100 [01:06<00:00,  1.50it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.4: 100%|██████████| 100/100 [01:11<00:00,  1.39it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.5: 100%|██████████| 100/100 [00:52<00:00,  1.89it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.6: 100%|██████████| 100/100 [00:52<00:00,  1.90it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.7: 100%|██████████| 100/100 [00:48<00:00,  2.07it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.8: 100%|██████████| 100/100 [00:47<00:00,  2.10it/s]


...

lr_sba_ROARL1_lamb0.1_new.pickle is saved!


Running lr_sba with lambda = 0.2, alpha = 0.8: 100%|██████████| 100/100 [00:24<00:00,  4.09it/s]


lr_sba_ROARL1_lamb0.2_new.pickle is saved!


Running lr_sba with lambda = 0.3, alpha = 0.8: 100%|██████████| 100/100 [00:18<00:00,  5.32it/s]

lr_sba_ROARL1_lamb0.3_new.pickle is saved!
